# Solutions — Performance and Best Practices (Hard 11)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_lineitem     = spark.table("samples.tpch.lineitem")
df_orders       = spark.table("samples.tpch.orders")
df_customer     = spark.table("samples.tpch.customer")
df_nation       = spark.table("samples.tpch.nation")
df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_franchises   = spark.table("samples.bakehouse.sales_franchises")


## Solution 1 — Cache and Unpersist

In [ ]:
# 1. Cache
df_lineitem.cache()
# 2. Trigger materialisation
row_count = df_lineitem.count()
cached = df_lineitem.is_cached
# 3. Unpersist
df_lineitem.unpersist()

result_1 = spark.createDataFrame(
    [
        ("1_cache",       "df_lineitem.cache()",         "cached"),
        ("2_count",       "df_lineitem.count()",         str(row_count)),
        ("3_is_cached",   "df_lineitem.is_cached",       str(cached)),
        ("4_unpersist",   "df_lineitem.unpersist()",     "evicted"),
    ],
    ["step", "description", "value"]
)


## Solution 2 — repartition vs coalesce

In [ ]:
default_partitions     = df_transactions.rdd.getNumPartitions()
after_repartition      = df_transactions.repartition(8).rdd.getNumPartitions()
after_coalesce         = df_transactions.repartition(8).coalesce(2).rdd.getNumPartitions()

result_2 = spark.createDataFrame(
    [
        ("default",       default_partitions),
        ("repartition(8)", after_repartition),
        ("coalesce(2)",   after_coalesce),
    ],
    ["operation", "partition_count"]
)


## Solution 3 — Broadcast Join for Small Tables

In [ ]:
# df_nation is small enough to broadcast
result_3 = (
    df_customer
    .join(F.broadcast(df_nation),
          df_customer.c_nationkey == df_nation.n_nationkey)
    .select(
        F.col("c_custkey"),
        F.col("n_name"),
    )
    .limit(1000)
)


## Solution 4 — Reading the Execution Plan

In [ ]:
result_4 = (
    df_transactions
    .join(df_franchises, on="franchiseID")
    .groupBy("franchiseID", "product")
    .agg(F.round(F.sum("totalPrice"), 2).alias("total_revenue"))
)
# result_4.explain(extended=True)  # uncomment to see full plan


## Solution 5 — Predicate Pushdown: Filter Early

In [ ]:
# Filter BEFORE join — Spark's Catalyst will push this down anyway,
# but being explicit is clearer and safer.
high_value = df_franchises.filter(F.col("size") == "Large")

result_5 = (
    df_transactions
    .join(high_value, on="franchiseID")
    .groupBy("franchiseID")
    .agg(F.round(F.sum("totalPrice"), 2).alias("total_revenue"))
)


## Solution 6 — Co-partitioning Before a Join

In [ ]:
# Repartition both DataFrames on the join key to avoid a shuffle
tx_part = df_transactions.repartition(8, "franchiseID")
fr_part = df_franchises.repartition(8, "franchiseID")

result_6 = (
    tx_part
    .join(fr_part, on="franchiseID")
    .groupBy("franchiseID", F.col("name"))
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("total_revenue"),
        F.count("*").alias("transaction_count"),
    )
)


## Solution 7 — Never collect() Large DataFrames

In [ ]:
# Use agg() instead of collect() for summaries
max_date = df_orders.agg(F.max("o_orderdate")).collect()[0][0]

result_7 = spark.createDataFrame(
    [("agg+collect single value", str(max_date))],
    ["approach", "max_order_date"]
)


## Solution 8 — Salting for Skewed Joins

In [ ]:
# Simulate a skewed key by adding a salt column
import random

NUM_SALT = 4

# Salt the small side: replicate rows for each salt value
salted_nation = df_nation.withColumn(
    "salt", F.explode(F.array([F.lit(i) for i in range(NUM_SALT)]))
)

# Salt the large side: assign a random salt value
salted_lineitem = (
    df_lineitem
    .withColumn("salt", (F.rand() * NUM_SALT).cast("int"))
    .join(df_orders.select("o_orderkey", "o_custkey"),
          F.col("l_orderkey") == F.col("o_orderkey"))
    .join(df_customer.select("c_custkey", "c_nationkey"),
          F.col("o_custkey") == F.col("c_custkey"))
    .withColumn("join_key", F.concat_ws("_", F.col("c_nationkey").cast("string"), F.col("salt").cast("string")))
)

salted_nation2 = salted_nation.withColumn(
    "join_key", F.concat_ws("_", F.col("n_nationkey").cast("string"), F.col("salt").cast("string"))
)

result_8 = (
    salted_lineitem
    .join(salted_nation2, on="join_key")
    .groupBy("l_shipmode", "n_name")
    .agg(
        F.round(F.sum("l_extendedprice"), 2).alias("total_revenue"),
        F.count("*").alias("line_count"),
    )
    .drop("n_name")
    .groupBy("l_shipmode")
    .agg(
        F.round(F.sum("total_revenue"), 2).alias("total_revenue"),
        F.sum("line_count").alias("line_count"),
    )
    .limit(1)
)
